# Lesson 14 — 让局域网工具发现提醒器

这节课仍然延续桌面状态提醒器，但数据首次穿过**真实本机网络**。不需要 ESP32，也不需要 Python 基础：下面代码格是完整 C++ 程序。

## 我们要解决什么问题？

当还不知道设备 IP 时，一个小工具怎样广播“谁在线”？

## 最小知识

UDP 以 datagram 为单位交付：本地的一次 `sendto` 对应接收方的一次 `recvfrom`。它不保证送达、顺序或重试；本课只验证 localhost 上实际 datagram 的边界。

运行初始化格后，所有网络实验只绑定 `127.0.0.1`。Notebook 面板是真实客户端；业务逻辑、协议响应和状态都来自当前 C++ 代码格。

In [ ]:
# 运行支架：只执行一次。你写的网络逻辑仍是 C++，不需要学习 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))
from network_lab import register
register()


## 1. 第一个可运行实验

点击“发送 DISCOVER”。看原始 datagram。这里的自动检查只确认本机真实 UDP 往返，不会虚构无线丢包；丢包、重试与设备发现策略将在真实网络环境继续验证。

运行下一格。面板的端口由 C++ 程序通过 `ACADEMY_READY` 报告；不要手填固定端口。

In [ ]:
%%network_lab l14-guided udp guided pass
#include "academy/net/socket.hpp"

int main() {
    return academy::net::run_udp_server([](std::string_view packet) {
        if (packet == "DISCOVER") return std::string("DESK_REMINDER|ONLINE");
        return std::string("ERR");
    });
}


## 2. 修改一次

把下一格中一个响应文本或状态初值改掉，再点击面板操作，观察改变从 C++ 穿过真实 localhost 连接回到输出。重新运行本格会终止旧进程并启动新进程。

In [ ]:
%%network_lab l14-modify udp
#include "academy/net/socket.hpp"

int main() {
    return academy::net::run_udp_server([](std::string_view packet) {
        if (packet == "DISCOVER") return std::string("DESK_REMINDER|ONLINE");
        return std::string("ERR");
    });
}


## 3. 故意弄坏

下面代码可编译，但**自动检查应失败**。先运行，点击“播放自动检查”，再根据红色结果解释问题。失败来自真实网络往返后的行为差异，不是 Notebook 预设动画。

In [ ]:
%%network_lab l14-broken udp exercise fail
#include "academy/net/socket.hpp"

int main() {
    return academy::net::run_udp_server([](std::string_view packet) {
        if (packet == "DISCOVER") return std::string("DESK_REMINDER|ONLINE\n");
        return std::string("ERR");
    });
}


## 4. 自己完成

在下方 C++ 中完成 TODO。先手动操作，再点“播放自动检查”。测试只编译这一格，不会读取 `solution/`。需要时再看 [分层 Hint](../lessons/14-udp-discovery/hints.md)。

In [ ]:
%%network_lab l14-exercise udp exercise
#include "academy/net/socket.hpp"

int main() {
    return academy::net::run_udp_server([](std::string_view packet) {
        // TODO: DISCOVER 必须回复 DESK_REMINDER|ONLINE，不能多换行或额外字段。
        (void)packet;
        return std::string("TODO");
    });
}


## 5. 你刚刚真正学到了什么？

你已经让提醒器通过真实 localhost 网络与另一个进程通信，并使用协议契约验证状态。迁移到 ESP32 后，socket API 会换为 ESP-IDF/lwIP 适配；发现报文的格式、超时和上层业务逻辑可保留。广播地址、AP 隔离和手机网络权限必须在实网检查。

完成尝试后才查看 [Solution Notebook](solutions/14-udp-discovery-solution.ipynb)。